In [1]:
import random,os
import pandas as pd
import numpy as np
import missingno as msno
from Data_cleaning_utils import perform_data_cleaning
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn import set_config
set_config(transform_output='pandas')
import optuna as optuna


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 0
seed_everything(seed)

import dagshub
dagshub.init(repo_owner='shapniljoy', repo_name='delivery-time-prediction', mlflow=True)

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow")
mlflow.set_experiment('Random Forest Hyperparameter Tuning')

Accessing as shapniljoy

Initialized MLflow to track repo "shapniljoy/delivery-time-prediction"

Repository shapniljoy/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/c4bca2fa7a064743ae83edd5ca6ba92d', creation_time=1782057513753, experiment_id='3', last_update_time=1782057513753, lifecycle_stage='active', name='Random Forest Hyperparameter Tuning', tags={}, trace_location=None, workspace='default'>

In [11]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','age', 'ratings','distance_km','pickup_time'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (36401, 17) and (36401,) 

Testing data: (9101, 17) and (9101,) 



In [12]:
age_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(drop='first',sparse_output=False))
])

ratings_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OE', OrdinalEncoder(categories=[['less than 4','4-4.5','4.5-5']],
                handle_unknown='use_encoded_value', unknown_value=-999))
])

distance_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OE', OrdinalEncoder(categories=[['short','medium','long','very_long']],
                handle_unknown='use_encoded_value', unknown_value=-999))
])

weather_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(drop='first',sparse_output=False))
])

traffic_missing = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(drop='first',sparse_output=False))
])

festival_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(drop='first',sparse_output=False))
])

city_type_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OE', OrdinalEncoder(categories=[['Semi-Urban','Urban','Metropolitan']],
    handle_unknown='use_encoded_value', unknown_value=-999))
])

time_of_day_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(drop='first',sparse_output=False))
])

pickup_time_missing = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('OE', OrdinalEncoder(categories=[['5 minutes','10 minutes','15 minutes']],
                          handle_unknown='use_encoded_value', unknown_value=-999))
])


preprocessor = ColumnTransformer([
        ('age', age_missing, ['age_cat']),
        ('ratings', ratings_missing, ['ratings_cat']),
        ('distance', distance_missing, ['distance_km_cat']),
        ('weather', weather_missing, ['weather']),
        ('traffic', traffic_missing, ['traffic']),
        ('festival', festival_missing, ['festival']),
        ('city_type', city_type_missing, ['city_type']),
        ('time_of_day', time_of_day_missing, ['time_of_day']),
        ('pickup_time', pickup_time_missing,['pickup_time_cat']),
        ('multiple_deliveries', SimpleImputer(strategy='most_frequent'), ['multiple_deliveries']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week']),
        ],remainder='passthrough')

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('age',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('OHE',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False))]),
                                 ['age_cat']),
                                ('ratings',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('OE',
                                                  OrdinalEncoder(categories=[['less '
                                                                              'than '
                                                                              '4',
                                                                              '4-4.5',
                                                                              '4.5-5']],
                                                                 h...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('OE',
                                                  OrdinalEncoder(categories=[['5 '
                                                                              'minutes',
                                                                              '10 '
                                                                              'minutes',
                                                                              '15 '
                                                                              'minutes']],
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-999))]),
                                 ['pickup_time_cat']),
                                ('multiple_deliveries',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['multiple_deliveries']),
                                ('OHE',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['vehicle_type', 'order_type',
                                  'day_of_week'])])

In [14]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        # model_name = trial.suggest_categorical('model',['RF'])

        # if model_name == 'RF':
        rf_params = { 
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth' : trial.suggest_int('max_depth', 4, 20),
        'min_samples_split' : trial.suggest_int('min_samples_split', 5, 10),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 5, 10),
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'max_samples' : trial.suggest_float('max_samples', 0.5, 1.0) 
            }
            
            
        model = RandomForestRegressor(
                **rf_params,
                bootstrap=True,
                random_state=seed,
                verbose=False,
                n_jobs=-1
            )


    # x_train_processed = preprocessor.fit_transform(x_train)
    # x_test_processed = preprocessor.transform(x_test)

        preprocessor = ColumnTransformer([
            ('age', age_missing, ['age_cat']),
            ('ratings', ratings_missing, ['ratings_cat']),
            ('distance', distance_missing, ['distance_km_cat']),
            ('weather', weather_missing, ['weather']),
            ('traffic', traffic_missing, ['traffic']),
            ('festival', festival_missing, ['festival']),
            ('city_type', city_type_missing, ['city_type']),
            ('time_of_day', time_of_day_missing, ['time_of_day']),
            ('pickup_time', pickup_time_missing,['pickup_time_cat']),
            ('multiple_deliveries', SimpleImputer(strategy='most_frequent'), ['multiple_deliveries']),
            ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week']),
            ],remainder='passthrough')

    
        # model = TransformedTargetRegressor(regressor=base_model,func=np.log1p, inverse_func=np.expm1) 

        model = Pipeline([
                ('preprocessor',preprocessor),
                ('model',model)
            ])

        cv_score = cross_validate(model,x_train,y_train,scoring='neg_mean_absolute_error',
                                    cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=1,verbose=False,return_train_score=True)

        trial.set_user_attr('train_cv',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_cv',np.mean(cv_score['test_score']))

        mlflow.log_metric('train_cv_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_cv_score',np.mean(cv_score['test_score']))

        return np.mean(cv_score['test_score'])


study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='Random Forest Hyperparameter Tuning')

with mlflow.start_run(run_name='Best MAE Score_2') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('best_mae_score_2',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value, '\n')


[I 2026-06-21 23:19:57,786] A new study created in memory with name: Random Forest Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/93068bbe6f114a30878367c7eec6526b
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-21 23:20:16,412] Trial 0 finished with value: -4.183475789160248 and parameters: {'n_estimators': 594, 'max_depth': 16, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.9458865003910399}. Best is trial 0 with value: -4.183475789160248.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/132049e69dac4dd38e6f0efe9d3ea27f
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-21 23:20:31,054] Trial 1 finished with value: -4.527693821359907 and parameters: {'n_estimators': 968, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.5435646498507

In [15]:
best_model = RandomForestRegressor(**study.best_params,
                bootstrap=True,
                random_state=seed,
                verbose=False)

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

final_rf_model = best_model.fit(x_train_processed,y_train)

# final_cat_model =TransformedTargetRegressor(regressor=best_model,func=np.log1p, inverse_func=np.expm1).fit(x_train,y_train)

y_pred_train = final_rf_model.predict(x_train_processed)
y_pred_test = final_rf_model.predict(x_test_processed)

print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
print(f"Training R2 score: {r2_score(y_test,y_pred_test)}")
print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")


mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
mlflow.log_metric('Training R2 score',r2_score(y_test,y_pred_test))
mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

mlflow.sklearn.log_model(best_model,name='RF_model_2')






Training error: 3.378394487584066
Testing error: 3.7178562946451796
Training R2 score: 0.7381792599605603
Testing R2 score: 0.7381792599605603


2026/06/21 23:42:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


# Optuna After Dropping Missing Values

In [20]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','age',
        'ratings','distance_km','pickup_time','day','month'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [21]:
rating_order = ['less than 4','4-4.5','4.5-5']

distance_order = ['short','medium','long','very_long']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

pickup_time_order = ['5 minutes','10 minutes','15 minutes']

ordinal_encoding = OrdinalEncoder(categories=[rating_order,distance_order,city_type_order,pickup_time_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['ratings_cat','distance_km_cat','city_type','pickup_time_cat']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'age_cat','weather','traffic','festival','time_of_day']),
        ],remainder='passthrough')

In [24]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        # model_name = trial.suggest_categorical('model',['RF'])

        # if model_name == 'RF':
        rf_params = { 
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth' : trial.suggest_int('max_depth', 4, 20),
        'min_samples_split' : trial.suggest_int('min_samples_split', 5, 10),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 5, 10),
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'max_samples' : trial.suggest_float('max_samples', 0.5, 1.0) 
            }
            
            
        model = RandomForestRegressor(
                **rf_params,
                bootstrap=True,
                random_state=seed,
                verbose=False,
                n_jobs=-1
            )


    # x_train_processed = preprocessor.fit_transform(x_train)
    # x_test_processed = preprocessor.transform(x_test)

        preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['ratings_cat','distance_km_cat','city_type','pickup_time_cat']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'age_cat','weather','traffic','festival','time_of_day']),
        ],remainder='passthrough')

    
        # model = TransformedTargetRegressor(regressor=base_model,func=np.log1p, inverse_func=np.expm1) 

        model = Pipeline([
                ('preprocessor',preprocessor),
                ('model',model)
            ])

        cv_score = cross_validate(model,x_train,y_train,scoring='neg_mean_absolute_error',
                                    cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=1,verbose=False,return_train_score=True)

        trial.set_user_attr('train_cv',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_cv',np.mean(cv_score['test_score']))

        mlflow.log_metric('train_cv_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_cv_score',np.mean(cv_score['test_score']))

        return np.mean(cv_score['test_score'])


if mlflow.active_run():
    mlflow.end_run()


study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='Random Forest Hyperparameter Tuning')

with mlflow.start_run(run_name='Dropped Missing') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('Dropped Missing',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value, '\n')


🏃 View run monumental-trout-561 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/004a4045186a4e108d0591c2dc792578
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3


[I 2026-06-22 00:18:19,932] A new study created in memory with name: Random Forest Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/8fa81df3f9c142809cb46185cf376d69
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-22 00:18:35,523] Trial 0 finished with value: -3.9819209896697094 and parameters: {'n_estimators': 594, 'max_depth': 16, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.9458865003910399}. Best is trial 0 with value: -3.9819209896697094.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/82ef567b690242929e1160a9779722a2
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-22 00:18:50,609] Trial 1 finished with value: -4.302586254002027 and parameters: {'n_estimators': 968, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.54356464985

In [27]:
best_model = RandomForestRegressor(**study.best_params,
                bootstrap=True,
                random_state=seed,
                verbose=False)

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

final_rf_model = best_model.fit(x_train_processed,y_train)

# final_cat_model =TransformedTargetRegressor(regressor=best_model,func=np.log1p, inverse_func=np.expm1).fit(x_train,y_train)

y_pred_train = final_rf_model.predict(x_train_processed)
y_pred_test = final_rf_model.predict(x_test_processed)

print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
print(f"Training R2 score: {r2_score(y_train,y_pred_train)}")
print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")


mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
mlflow.log_metric('Training R2 score',r2_score(y_train,y_pred_train))
mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

mlflow.sklearn.log_model(best_model,name='Dropped Missing')






Training error: 3.2280202686431676
Testing error: 3.5291978993033983
Training R2 score: 0.8036317975834699
Testing R2 score: 0.7697788260001486


2026/06/22 01:30:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


# Optuna After dropping missing values and With Numerical variables

In [2]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','day','month',
        'age_cat','ratings_cat','distance_km_cat','pickup_time_cat'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [3]:

numerical_cols = ['age','ratings','distance_km','pickup_time']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

ordinal_encoding = OrdinalEncoder(categories=[city_type_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['city_type']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'weather','traffic','festival','time_of_day']),
        ('scaling',StandardScaler(),numerical_cols),
        ],remainder='passthrough')

In [4]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        # model_name = trial.suggest_categorical('model',['RF'])

        # if model_name == 'RF':
        rf_params = { 
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth' : trial.suggest_int('max_depth', 4, 20),
        'min_samples_split' : trial.suggest_int('min_samples_split', 5, 10),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 5, 10),
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'max_samples' : trial.suggest_float('max_samples', 0.5, 1.0) 
            }
            
            
        model = RandomForestRegressor(
                **rf_params,
                bootstrap=True,
                random_state=seed,
                verbose=False,
                n_jobs=-1
            )
    
        # model = TransformedTargetRegressor(regressor=base_model,func=np.log1p, inverse_func=np.expm1) 

        model = Pipeline([
                ('preprocessor',preprocessor),
                ('model',model)
            ])

        cv_score = cross_validate(model,x_train,y_train,scoring='neg_mean_absolute_error',
                                    cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=1,verbose=False,return_train_score=True)

        trial.set_user_attr('train_cv',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_cv',np.mean(cv_score['test_score']))

        mlflow.log_metric('train_cv_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_cv_score',np.mean(cv_score['test_score']))

        return np.mean(cv_score['test_score'])


if mlflow.active_run():
    mlflow.end_run()


study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='Random Forest Hyperparameter Tuning')

with mlflow.start_run(run_name='With Numerical cols') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('With Numerical cols',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value)


[I 2026-06-22 16:13:44,302] A new study created in memory with name: Random Forest Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/c94469d1952644569f8b44327ce44d06
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-22 16:14:05,158] Trial 0 finished with value: -3.5589187449293376 and parameters: {'n_estimators': 594, 'max_depth': 16, 'min_samples_split': 8, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.9458865003910399}. Best is trial 0 with value: -3.5589187449293376.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3/runs/4be949044a8d4dd5ba39a38db4c22087
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/3
[I 2026-06-22 16:14:21,651] Trial 1 finished with value: -3.9116274755955254 and parameters: {'n_estimators': 968, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.5435646498

In [8]:
study.trials_dataframe()[['user_attrs_train_cv','user_attrs_val_cv']].sort_values(by='user_attrs_val_cv',ascending=False)

,user_attrs_train_cv,user_attrs_val_cv
24,-2.599609,-3.072460
22,-2.533783,-3.074398
11,-2.650676,-3.074802
23,-2.595479,-3.075044
27,-2.690251,-3.075872
21,-2.497542,-3.076112
20,-2.471719,-3.076560
15,-2.528824,-3.076763
16,-2.535586,-3.076830
17,-2.507301,-3.077252


In [9]:
study.best_params

{'n_estimators': 902,
 'max_depth': 14,
 'min_samples_split': 10,
 'min_samples_leaf': 7,
 'max_features': None,
 'max_samples': 0.9857978714826628}

In [10]:
best_model = RandomForestRegressor(**study.best_params,
                bootstrap=True,
                random_state=seed,
                verbose=False)

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

final_rf_model = best_model.fit(x_train_processed,y_train)

# final_cat_model =TransformedTargetRegressor(regressor=best_model,func=np.log1p, inverse_func=np.expm1).fit(x_train,y_train)

y_pred_train = final_rf_model.predict(x_train_processed)
y_pred_test = final_rf_model.predict(x_test_processed)

print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
print(f"Training R2 score: {r2_score(y_train,y_pred_train)}")
print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")


mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
mlflow.log_metric('Training R2 score',r2_score(y_train,y_pred_train))
mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

mlflow.sklearn.log_model(best_model,name='With Numerical cols')






Training error: 2.6170472615172935
Testing error: 3.0245296897272653
Training R2 score: 0.8791414071956365
Testing R2 score: 0.8385149998431674


2026/06/22 16:33:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
